# Experiment K: Verbalizer (Prompt-based Classification)

Instead of pooling hidden states through a classifier head, use the
pretrained MLM head as a prompt-based classifier.

**Approach:**
1. Frame the task as a QA cloze: `"Is this text patronising? {text} Answer: [MASK]"`
2. Pass through `AutoModelForMaskedLM` to get vocab logits at the `[MASK]` position
3. Score = `log P("Yes" | mask) − log P("No" | mask)`
4. Train with `BCEWithLogitsLoss` — the log-odds output is directly compatible

**Key advantages** of Yes/No verbalizers over adjective-based ones:
- "Yes" and "No" are guaranteed single tokens — no subword fragmentation
- The question template directly describes the classification task
- The MLM head's prior for Yes/No after "Answer:" is strong and well-calibrated

Hyperparameters searched: LR, warmup, weight decay, label smoothing,
verbalizer variant (Yes/No, yes/no, True/False), question phrasing.

In [ ]:
import os
import sys
import random
import logging
import gc
import json

import numpy as np
import torch
from sklearn.metrics import classification_report
from transformers import AutoTokenizer
import optuna
from optuna.visualization.matplotlib import (
    plot_optimization_history,
    plot_param_importances,
    plot_parallel_coordinate,
)
import matplotlib.pyplot as plt

sys.path.insert(0, "..")
from utils.data import load_data
from utils.split import split_train_val
from utils.pcl_dataset import PCLVerbalizerDataset
from utils.pcl_deberta import PCLDeBERTaVerbalizer
from utils.optim import compute_pos_weight
from utils.training_loop import train_model
from utils.eval import evaluate

SEED = 42
DATA_DIR = "../data"
OUT_DIR = "out"
MODEL_NAME = "microsoft/deberta-v3-base"
MAX_LENGTH = 256
VAL_FRACTION = 0.15
BATCH_SIZE = 32
N_TRIALS = 20
NUM_EPOCHS = 12
PATIENCE = 4
N_EVAL_STEPS = 35
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s:\t%(message)s")
LOG = logging.getLogger(__name__)
LOG.info(f"Device: {DEVICE}")
os.makedirs(OUT_DIR, exist_ok=True)

## 1. Data Loading

In [ ]:
train_df, dev_df = load_data(DATA_DIR)
train_sub_df, val_sub_df = split_train_val(train_df, val_frac=VAL_FRACTION, seed=SEED)
tokeniser = AutoTokenizer.from_pretrained(MODEL_NAME)
LOG.info(f"Train: {len(train_sub_df)}, Val: {len(val_sub_df)}, Dev: {len(dev_df)}")

## 2. Verbalizer & Template Setup

The verbalizer maps binary classes to Yes/No tokens.
The main hyperparameter is the **question phrasing** — how we frame
the PCL detection task as a cloze question for the MLM head.

Three verbalizer variants (casing/wording) × four question templates
are searched jointly by Optuna.

In [ ]:
from torch.utils.data import DataLoader

# --- Define verbalizer variants ---
# Each entry: (name, positive_words, negative_words)
# Yes/No are guaranteed single tokens in SentencePiece.
VERBALIZER_SETS = [
    ("Yes_No",     ["Yes"],  ["No"]),
    ("yes_no",     ["yes"],  ["no"]),
    ("True_False", ["True"], ["False"]),
    ("true_false", ["true"], ["false"]),
]

# --- Question-style templates ---
# The question phrasing is the main thing being searched.
# {text} = input paragraph, {mask} = single [MASK] token.
TEMPLATE_OPTIONS = [
    'Is the following text patronising or condescending? "{text}" Answer: {mask}',
    'Does the author of this text sound patronising or condescending? "{text}" Answer: {mask}',
    'Is this text talking down to people? "{text}" Answer: {mask}',
    'Does the following text contain patronising language? "{text}" Answer: {mask}',
]


def words_to_first_subword_ids(words: list[str]) -> list[int]:
    """Convert words to their first subword token IDs."""
    ids = []
    for w in words:
        subtokens = tokeniser.encode(w, add_special_tokens=False)
        ids.append(subtokens[0])
        n_sub = len(subtokens)
        decoded = tokeniser.decode([subtokens[0]])
        flag = " ✓ single token" if n_sub == 1 else f"  ⚠ {n_sub} subwords, using first ('{decoded}')"
        LOG.info(f"    '{w}' -> id {subtokens[0]}{flag}")
    return ids


# Pre-compute all verbalizer tokenisations and log diagnostics
VERBALIZER_CACHE: dict[str, tuple[list[int], list[int]]] = {}
for name, pos_words, neg_words in VERBALIZER_SETS:
    LOG.info(f"Verbalizer '{name}':")
    pos_ids = words_to_first_subword_ids(pos_words)
    neg_ids = words_to_first_subword_ids(neg_words)
    VERBALIZER_CACHE[name] = (pos_ids, neg_ids)

# Summary
print(f"\n{'Verbalizer':<12} pos_id  neg_id")
print("-" * 35)
for name, (p, n) in VERBALIZER_CACHE.items():
    print(f"{name:<12} {p}     {n}")

print(f"\nTemplates ({len(TEMPLATE_OPTIONS)}):")
for i, t in enumerate(TEMPLATE_OPTIONS):
    preview = t.format(text="...", mask="[MASK]")
    print(f"  [{i}] {preview}")


def make_verbalizer_dataloaders(
    template: str,
) -> tuple[DataLoader, DataLoader, DataLoader]:
    """Create DataLoaders with the verbalizer template applied (single [MASK])."""
    def make_ds(df):
        return PCLVerbalizerDataset(
            texts=df["text"].tolist(),
            labels=df["binary_label"].tolist(),
            max_length=MAX_LENGTH,
            tokeniser=tokeniser,
            template=template,
        )

    train_ds = make_ds(train_sub_df)
    val_ds = make_ds(val_sub_df)
    dev_ds = make_ds(dev_df)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=False)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
    dev_loader = DataLoader(dev_ds, batch_size=BATCH_SIZE, shuffle=False)

    return train_loader, val_loader, dev_loader

## 3. Hyperparameter Search

In [ ]:
EXP_NAME = "K_verbalizer"

# --- Fixed HPs (stable across prior experiments) ---
WARMUP_FRACTION = 0.06
LABEL_SMOOTHING = 0.0


def objective(trial: optuna.trial.Trial) -> float:
    # Searched HPs
    lr              = trial.suggest_float("lr", 4e-6, 6e-5, log=True)
    weight_decay    = trial.suggest_float("weight_decay", 5e-4, 5e-3, log=True)
    head_lr_mult    = trial.suggest_categorical("head_lr_multiplier", [1, 5, 10])
    verbalizer_name = trial.suggest_categorical("verbalizer", [v[0] for v in VERBALIZER_SETS])
    template_idx    = trial.suggest_categorical("template_idx", list(range(len(TEMPLATE_OPTIONS))))

    template = TEMPLATE_OPTIONS[template_idx]
    pos_ids, neg_ids = VERBALIZER_CACHE[verbalizer_name]

    LOG.info(f"[{EXP_NAME}] Trial {trial.number}: lr={lr:.2e}, "
             f"verbalizer={verbalizer_name}, template={template_idx}")

    # Dataloaders only depend on template (single [MASK]); verbalizer IDs go to the model
    train_loader, val_loader, dev_loader = make_verbalizer_dataloaders(template)

    model = PCLDeBERTaVerbalizer(
        pos_verbalizer_ids=pos_ids,
        neg_verbalizer_ids=neg_ids,
        model_name=MODEL_NAME,
    ).to(DEVICE)

    pos_weight = compute_pos_weight(train_sub_df, DEVICE)

    results = train_model(
        model=model, device=DEVICE,
        train_loader=train_loader, val_loader=val_loader, dev_loader=dev_loader,
        pos_weight=pos_weight, lr=lr, weight_decay=weight_decay,
        num_epochs=NUM_EPOCHS, warmup_fraction=WARMUP_FRACTION,
        patience=PATIENCE, head_lr_multiplier=head_lr_mult,
        label_smoothing=LABEL_SMOOTHING, eval_every_n_steps=N_EVAL_STEPS,
        trial=trial,
    )

    trial.set_user_attr("best_val_f1",    results["best_val_f1"])
    trial.set_user_attr("best_threshold", results["best_threshold"])
    trial.set_user_attr("dev_f1",         results["dev_metrics"]["f1"])
    trial.set_user_attr("dev_precision",  results["dev_metrics"]["precision"])
    trial.set_user_attr("dev_recall",     results["dev_metrics"]["recall"])

    try:
        prev_best = trial.study.best_value
    except ValueError:
        prev_best = -float("inf")
    if results["best_val_f1"] > prev_best:
        torch.save(
            {k: v.cpu() for k, v in model.state_dict().items()},
            os.path.join(OUT_DIR, f"exp_{EXP_NAME}_best_model.pt")
        )
        config = {**trial.params, "batch_size": BATCH_SIZE, "num_epochs": NUM_EPOCHS,
                  "patience": PATIENCE, "best_threshold": results["best_threshold"],
                  "warmup_fraction": WARMUP_FRACTION, "label_smoothing": LABEL_SMOOTHING}
        with open(os.path.join(OUT_DIR, f"exp_{EXP_NAME}_best_params.json"), "w") as f:
            json.dump(config, f, indent=2)
        LOG.info(f"[{EXP_NAME}] New best saved (val F1={results['best_val_f1']:.4f})")

    del model, train_loader, val_loader, dev_loader
    gc.collect()
    torch.cuda.empty_cache()
    return results["best_val_f1"]

## 4. Run Experiment

In [ ]:
gc.collect()
torch.cuda.empty_cache()

study = optuna.create_study(
    direction="maximize",
    study_name=f"pcl_deberta_exp_{EXP_NAME}",
    sampler=optuna.samplers.TPESampler(seed=SEED),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=6, n_warmup_steps=300),
)
study.optimize(objective, n_trials=N_TRIALS)

best = study.best_trial
LOG.info(f"Best trial: {best.number}")
LOG.info(f"Val F1: {best.user_attrs['best_val_f1']:.4f} | Dev F1: {best.user_attrs['dev_f1']:.4f}")
LOG.info(f"Best params: {best.params}")

## 5. Results

In [ ]:
for plot_fn, suffix in [
    (plot_optimization_history, "history"),
    (plot_param_importances, "importances"),
    (plot_parallel_coordinate, "parallel"),
]:
    plot_fn(study)
    plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/{EXP_NAME}_optuna_{suffix}.png", dpi=300)
    plt.show()

# Reload best model and evaluate on dev set
best = study.best_trial
best_params = best.params
verbalizer_name = best_params["verbalizer"]
template = TEMPLATE_OPTIONS[best_params["template_idx"]]
pos_ids, neg_ids = VERBALIZER_CACHE[verbalizer_name]

model = PCLDeBERTaVerbalizer(
    pos_verbalizer_ids=pos_ids,
    neg_verbalizer_ids=neg_ids,
    model_name=MODEL_NAME,
).to(DEVICE)

state_dict = torch.load(
    os.path.join(OUT_DIR, f"exp_{EXP_NAME}_best_model.pt"), map_location=DEVICE
)
model.load_state_dict(state_dict)

_, _, dev_loader = make_verbalizer_dataloaders(template)
dev_metrics = evaluate(model, DEVICE, dev_loader, threshold=best.user_attrs["best_threshold"])

print(f"\n{'='*60}")
print(f"{EXP_NAME.upper()} — Dev Set Results (threshold={best.user_attrs['best_threshold']:.3f})")
print(f"{'='*60}")
print(classification_report(dev_metrics["labels"], dev_metrics["preds"], target_names=["Non-PCL", "PCL"]))

# Show verbalizer details
verb_set = [v for v in VERBALIZER_SETS if v[0] == verbalizer_name][0]
print(f"  Verbalizer: {verbalizer_name}")
print(f"    pos words: {verb_set[1]}  -> ids {pos_ids}")
print(f"    neg words: {verb_set[2]}  -> ids {neg_ids}")
print(f"  Template: \"{template}\"")
for param_k, param_v in best_params.items():
    print(f"  {param_k}: {param_v}")

del model
gc.collect()
torch.cuda.empty_cache()